# Lecture 5: Computer-Assisted Discovery and Open Coding

This notebook begins from an already-cleaned CSV so the analysis can start with BERTopic.

It is written as a teaching notebook, so the code cells now use **line-by-line annotation**.
Each executable line is paired with a comment that explains what that precise line does.


## Workflow

1. Load the cleaned corpus.
2. Run BERTopic.
3. Inspect topics.
4. Plot embeddings with topic colors.
5. Export a reading sample for manual open coding.
6. Use Word2Vec to expand search terms.
7. Optionally run local Ollama open coding if Ollama is installed.
8. Run thematic analysis across coded posts.

## Environment note

This notebook expects these Python packages to be installed in the active `python3` environment:

- `pandas`
- `scikit-learn`
- `bertopic`
- `sentence-transformers`
- `umap-learn`
- `hdbscan`
- `plotly`
- `gensim`
- `requests`

The Ollama section is optional and will skip cleanly if `ollama` is not installed locally.


In [ ]:
import sys
print(sys.executable)



In [ ]:
# ── STUDENT CONFIGURATION ─────────────────────────────────────────────────────
# Change CORPUS_PATH to point to your own CSV file.
# Your CSV must have at least these columns: doc_id, link, text
# Optional but used if present: title, body, text_length
from pathlib import Path
import json, os, re, shutil

CORPUS_PATH = Path('your_corpus.csv')   # <-- CHANGE THIS

# Output folder: notebook will create it automatically.
OUTPUT_DIR = Path('notebook_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fixed random seed for reproducibility.
SEED = 21

os.environ['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'
os.environ['MPLCONFIGDIR']    = '/tmp/mplconfig'
Path('/tmp/numba_cache').mkdir(parents=True, exist_ok=True)
Path('/tmp/mplconfig').mkdir(parents=True, exist_ok=True)

print('Corpus path:', CORPUS_PATH)
print('Corpus exists:', CORPUS_PATH.exists())
print('Output dir:', OUTPUT_DIR)


In [ ]:
# Import pandas for tabular data handling.
import pandas as pd

# Limit the number of documents so the demo runs in a reasonable amount of time.
MAX_DOCS = 10000
# Read the cleaned CSV into a dataframe.
corpus_df = pd.read_csv(CORPUS_PATH)
# Keep only the columns needed in the lecture notebook.
corpus_df = corpus_df[['doc_id', 'link', 'title', 'body', 'text', 'text_length']].copy()
# Keep only the first MAX_DOCS rows for a lighter in-class workflow.
corpus_df = corpus_df.head(MAX_DOCS).copy()
# Reset the dataframe index after subsetting.
corpus_df = corpus_df.reset_index(drop=True)

# Print the dataframe shape to confirm how many rows and columns were loaded.
print(corpus_df.shape)
# Display the first few rows so the structure of the cleaned corpus is visible.
corpus_df.head(6)


## BERTopic as a discovery tool

BERTopic is used to orient reading and suggest possible categories. It is not treated as a final classifier.


In [ ]:
# Import CountVectorizer so BERTopic can build topic representations from word counts.
from sklearn.feature_extraction.text import CountVectorizer
# Import TfidfVectorizer so a local fallback embedding space can be built without internet access.
from sklearn.feature_extraction.text import TfidfVectorizer
# Import TruncatedSVD so sparse tf-idf vectors can be turned into dense low-dimensional embeddings.
from sklearn.decomposition import TruncatedSVD
# Import normalize so fallback embeddings are length-normalized before clustering.
from sklearn.preprocessing import normalize
# Import SentenceTransformer so documents can be embedded as vectors when the model is locally available.
from sentence_transformers import SentenceTransformer
# Import UMAP so embeddings can be reduced before clustering.
from umap import UMAP
# Import HDBSCAN so dense regions in embedding space can be clustered.
from hdbscan import HDBSCAN
# Import BERTopic as the main topic-modeling interface.
from bertopic import BERTopic

# Extract the document texts from the dataframe as a Python list.
texts = corpus_df['text'].tolist()
# Set the transformer model name used for the first embedding attempt.
embedding_model_name = 'all-MiniLM-L6-v2'
# Build the expected Hugging Face cache directory for this model.
transformer_cache_dir = Path.home() / '.cache' / 'huggingface' / 'hub' / 'models--sentence-transformers--all-MiniLM-L6-v2'
# Start with no embedding model until one is successfully created.
embedding_model = None
# Start with no embeddings until they are successfully created.
embeddings = None
# Start with a label that describes which embedding strategy was used.
embedding_strategy = None

# Check whether the transformer model appears to be cached locally.
if transformer_cache_dir.exists():
    # Load the sentence-transformer model from local files because a cache directory is available.
    embedding_model = SentenceTransformer(embedding_model_name, local_files_only=True)
    # Encode the texts into dense semantic embeddings.
    embeddings = embedding_model.encode(texts, show_progress_bar=True)
    # Record that transformer embeddings were used.
    embedding_strategy = 'sentence-transformer'
else:
    # Print a short message explaining why the notebook is using the local fallback path.
    print('Transformer model not cached locally, so the notebook is using tf-idf + SVD embeddings instead.')
    # Build a tf-idf vectorizer for a local document-feature matrix.
    fallback_vectorizer = TfidfVectorizer(stop_words='english', min_df=5, max_features=5000, ngram_range=(1, 2))
    # Fit the tf-idf vectorizer and transform the texts into a sparse matrix.
    tfidf_matrix = fallback_vectorizer.fit_transform(texts)
    # Build an SVD model that compresses the sparse tf-idf matrix into dense components.
    svd_model = TruncatedSVD(n_components=50, random_state=SEED)
    # Fit the SVD model and transform the tf-idf matrix into dense fallback embeddings.
    embeddings = svd_model.fit_transform(tfidf_matrix)
    # Normalize the fallback embeddings so distance calculations are more stable.
    embeddings = normalize(embeddings)
    # Record that tf-idf plus SVD embeddings were used.
    embedding_strategy = 'tfidf-svd'

# Create a UMAP model that reduces embedding dimensions before clustering.
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=SEED)
# Create an HDBSCAN model that clusters documents in the reduced space.
hdbscan_model = HDBSCAN(min_cluster_size=20, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# Create a vectorizer so topic labels can be summarized from common words and bigrams.
vectorizer_model = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))
# Create the BERTopic model from the reduction, clustering, and vectorization components.
topic_model = BERTopic(embedding_model=embedding_model, umap_model=umap_model, hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model, calculate_probabilities=True, verbose=True)
# Fit BERTopic to the texts and embeddings and collect the assigned topic for each document.
topics, probabilities = topic_model.fit_transform(texts, embeddings)
# Store the assigned topic for each document back in the dataframe.
corpus_df['topic'] = topics

# Print which embedding strategy was ultimately used.
print('embedding strategy:', embedding_strategy)
# Show the first few document ids together with their topic assignments.
corpus_df[['doc_id', 'topic']].head()


In [ ]:
# Ask BERTopic for a summary table of discovered topics. 
topic_info = topic_model.get_topic_info()
# Show the first rows of the topic summary so topic sizes and labels can be inspected.

for i in topic_info['Representation'].head(20):
    print(i)


In [ ]:
import plotly.express as px
from umap import UMAP

# Create a 2D UMAP model for plotting document embeddings.
plot_umap = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=SEED)
embedding_2d = plot_umap.fit_transform(embeddings)

plot_df = corpus_df[['doc_id', 'link', 'text', 'topic']].copy()
plot_df['x'] = embedding_2d[:, 0]
plot_df['y'] = embedding_2d[:, 1]
plot_df['topic_label'] = plot_df['topic'].astype(str)
plot_df['preview'] = plot_df['text'].str.slice(0, 220)

fig = px.scatter(plot_df, x='x', y='y', color='topic_label', hover_data=['doc_id', 'preview', 'link'], title='Embedding space colored by BERTopic assignment', opacity=0.75, width=1100, height=800)
fig.update_traces(marker={'size': 7})
fig.write_html(OUTPUT_DIR / 'bertopic_embedding_topics.html')
fig

In [ ]:
# Define a helper function that returns the top words for a topic.
def top_terms_for_topic(topic_id: int, n: int = 10) -> list[str]:
    # Return a placeholder label if the document belongs to the outlier topic.
    if topic_id == -1:
        return ['outlier / no stable cluster']
    # Otherwise return the top n terms associated with the topic.
    return [term for term, _ in topic_model.get_topic(topic_id)[:n]]

# Count how many documents belong to each topic.
topic_summary = corpus_df['topic'].value_counts().rename_axis('topic').reset_index(name='n_docs')
# Add a column showing the top terms for each topic.
topic_summary['top_terms'] = topic_summary['topic'].apply(top_terms_for_topic)
# Display the compact topic summary.
topic_summary.head(12)


# Intial list of categories and matching words

The categories below are my starting from reading across the top words of the topic modeling output. These words I can expand with word2vec in the next step. 

In [ ]:
objects_of_concern = {

    "partner": ["husband", "wife", "boyfriend", "girlfriend", "bf", "gf", "partner"],

    "ex_partner": ["ex", "broke", "months", "ago"],

    "friend": ["friend", "friends", "best friend", "friendship"],

    "family": ["mom", "dad", "parents", "sister", "brother", "mother", "father"],

    "third_parties_or_rivals": ["girls", "women", "photos", "videos", "porn"]

}

forms_of_concern = {

    "desire_or_infidelity": ["sex", "sexual", "porn", "watching", "girls", "women", "phone", "account"],

    "domestic_obligation": ["home", "house", "kids", "work", "bills", "money", "pay", "job"],

    "communication_breakdown": ["said", "told", "asked", "talked", "fight", "argue"],

    "emotional_uncertainty": ["feel", "feel like", "dont know", "doesnt", "want"],

    "separation_or_loss": ["ex", "broke", "divorce", "months", "ago"],

    "distance": ["long distance", "distance", "longdistance"]

}

## Sampling for open coding

After topics are discovered, the next step is to sample documents for reading rather than to treat the topics as final categories. This inorder to see if the topic model did return meaningfull topic which you might learn from. Either in the form of backgrund contextual knowlegde or as potential analytically relevant themes. 


In [ ]:
SAMPLE_PER_TOPIC = 5
# pandas 3.0 groupby.apply() drops the grouping column, so iterate explicitly.
sampled_reading_df = pd.concat([
    group.sample(min(len(group), SAMPLE_PER_TOPIC), random_state=SEED)
    for _, group in corpus_df.groupby('topic')
]).reset_index(drop=True)

sampled_reading_df['text_excerpt'] = sampled_reading_df['text'].str.slice(0, 500)
sampled_reading_df['topic_terms'] = sampled_reading_df['topic'].apply(lambda topic_id: ', '.join(top_terms_for_topic(topic_id, n=6)))
reading_export = sampled_reading_df[['doc_id', 'link', 'topic', 'topic_terms', 'text_excerpt']].copy()
reading_export.to_csv(OUTPUT_DIR / 'bertopic_open_coding_sample.csv', index=False)
reading_export.head(20)

## Word2Vec for search-term expansion

The final section uses word2vec to expand a seed list and retrieve related documents. 


In [ ]:
# Import Word2Vec so nearby terms can be learned from the corpus.
from gensim.models import Word2Vec

# Create a simple regular expression that keeps alphabetic tokens and contractions.
TOKEN_PATTERN = re.compile(r"[A-Za-z][A-Za-z']+")

# Define a small tokenizer that lowercases text and extracts tokens with the regex.
def simple_tokenize(text: str) -> list[str]:
    # Return the lowercase tokens found by the regular expression.
    return TOKEN_PATTERN.findall(text.lower())

# Tokenize every document in the corpus.
tokenized_docs = [simple_tokenize(text) for text in corpus_df['text']]
# Remove documents that are too short to be useful for Word2Vec.
tokenized_docs = [tokens for tokens in tokenized_docs if len(tokens) >= 5]
# Train a small skip-gram Word2Vec model on the tokenized corpus.
w2v_model = Word2Vec(sentences=tokenized_docs, vector_size=100, window=5, min_count=5, workers=1, sg=1, seed=SEED)

# Print how many documents were used to train the Word2Vec model.
print(f'Trained on {len(tokenized_docs)} documents')


In [ ]:
# Search for similar words or lists of words
w2v_model.wv.most_similar('ex',topn=15)

## Optional Ollama section

This section now skips cleanly if Ollama is not installed on the machine.


In [ ]:
import requests

OLLAMA_URL = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'gemma3:4b'

CODING_INSTRUCTIONS = """You are an open-ended qualitative coder analyzing Reddit posts about relationship troubles.

Your aim is to develop initial codes inductively from the material. Use the broad sensitizing concept of "relational trouble" to guide your reading, but do not impose fixed categories in advance.

As you read, ask:

* Who is the relational other, or alter, around whom the trouble is organized?
* What is the poster worried about, hurt by, confused by, or trying to resolve in that relationship?
* How does the poster formulate the trouble: as a violation, uncertainty, mismatch, conflict, unmet obligation, loss of trust, lack of care, desire for someone else, emotional distance, control, dependency, or something else?
* Are there multiple relational troubles layered together?

Produce open codes that capture both the relation involved and the substantive concern. Prefer codes such as "husband's emotional withdrawal," "boyfriend's possible desire for other women," "conflict with parents over boundaries," or "uncertainty after breakup" rather than broad topic labels such as "marriage," "porn," or "family."

Do not decide who is right. Do not offer advice. Do not summarize the whole post generically. Code how the relational trouble is articulated.

If the post is too short, spam, or clearly not about relational troubles, set irrelevant to true and leave other fields as empty strings.

Return ONLY valid JSON with exactly these five keys:
  relational_alters  : comma-separated list of the relational others involved
  main_troubles      : comma-separated list of the main relational troubles
  open_codes         : comma-separated initial open codes (prefer specific, relation-anchored phrases)
  memo               : 1-2 sentence analytic note on how the trouble is articulated
  irrelevant         : true or false""".strip()

OLLAMA_CANDIDATE_PATHS = ['/usr/local/bin/ollama', '/Applications/Ollama.app/Contents/Resources/ollama']
OLLAMA_BINARY = next((p for p in OLLAMA_CANDIDATE_PATHS if Path(p).exists()), None)

OLLAMA_AVAILABLE = False
if shutil.which('ollama') is not None or OLLAMA_BINARY is not None:
    try:
        requests.get('http://localhost:11434', timeout=2)
        OLLAMA_AVAILABLE = True
    except Exception:
        pass

def build_prompt(text: str) -> str:
    return f"{CODING_INSTRUCTIONS}\n\nText:\n{text}"

def call_ollama(prompt: str) -> dict:
    response = requests.post(
        OLLAMA_URL,
        json={'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False, 'format': 'json'},
        timeout=120,
    )
    response.raise_for_status()
    return json.loads(response.json()['response'])

print('ollama available:', OLLAMA_AVAILABLE)
print('model:', OLLAMA_MODEL)

In [ ]:
OPEN_CODE_N = 10

def coerce_str(val) -> str:
    if isinstance(val, list):
        return ', '.join(str(v) for v in val)
    return str(val) if val else ''

def parse_result(parsed: dict, row) -> dict:
    return {
        'doc_id':            row.doc_id,
        'topic':             row.topic,
        'topic_terms':       row.topic_terms,
        'relational_alters': coerce_str(parsed.get('relational_alters', '')),
        'main_troubles':     coerce_str(parsed.get('main_troubles', '')),
        'open_codes':        coerce_str(parsed.get('open_codes', '')),
        'memo':              parsed.get('memo', ''),
        'irrelevant':        bool(parsed.get('irrelevant') or False),
        'text_excerpt':      row.text_excerpt,
        'link':              row.link,
    }

if OLLAMA_AVAILABLE:
    open_coding_df = reading_export.head(OPEN_CODE_N).copy()
    results = []
    for row in open_coding_df.itertuples(index=False):
        parsed = call_ollama(build_prompt(row.text_excerpt))
        results.append(parse_result(parsed, row))

    open_coded_results_df = pd.DataFrame(results)
    open_coded_results_df.to_csv(OUTPUT_DIR / 'ollama_open_coding_results.csv', index=False)
    open_coded_results_df[['doc_id', 'relational_alters', 'main_troubles', 'open_codes', 'memo', 'irrelevant']]
else:
    print('Skipping Ollama coding — server not running. Start it with: ollama serve')

In [ ]:
THEMATIC_PROMPT = """You are a qualitative sociologist doing thematic analysis of Reddit relationship advice posts.

You have received open-coded material. Each entry is:
[doc_id] | alters: ... | codes: ... | memo: ...

Your task:
1. Identify 3–4 recurring analytical themes in how relational trouble is articulated.
2. Each theme should capture a distinctive mode or logic of trouble — not just a topic.
3. For each theme provide:
   - A short evocative title.
   - A one-sentence analytical summary of what the theme is.
   - 4–6 direct quotes from the posts that illustrate the theme. For each quote give: the doc_id, the exact quote (copy the wording from the memo or codes — do not paraphrase), and a one-sentence analytical memo explaining what it shows about how trouble is articulated.

Return ONLY valid JSON with this structure:
{
  "themes": [
    {
      "title": "short evocative title",
      "summary": "one analytical sentence about the theme",
      "examples": [
        {
          "doc_id": "doc_id_here",
          "quote": "exact wording from the material",
          "memo": "one-sentence analytic note"
        }
      ]
    }
  ]
}

Coded material:

"""

def build_thematic_material(df: pd.DataFrame, n: int = 25) -> str:
    lines = []
    for _, row in df.iterrows():
        if row.get('irrelevant'):
            continue
        lines.append(
            f"[{row['doc_id']}] | "
            f"alters: {row['relational_alters']} | "
            f"codes: {row['open_codes']} | "
            f"memo: {row['memo']}"
        )
        if len(lines) >= n:
            break
    return '\n'.join(lines)

def render_thematic_markdown(themes: list) -> str:
    lines = ['# Thematic Analysis\n']
    for i, theme in enumerate(themes, 1):
        lines.append(f'## Theme {i}: {theme["title"]}\n')
        lines.append(f'*{theme.get("summary", "")}*\n')
        for ex in theme.get('examples', []):
            doc = ex.get('doc_id', '')
            quote = ex.get('quote', '')
            memo = ex.get('memo', '')
            lines.append(f'> "{quote}"\n> `{doc}` — {memo}\n')
    return '\n'.join(lines)

if OLLAMA_AVAILABLE:
    coded_df = pd.read_csv(OUTPUT_DIR / 'ollama_open_coding_results.csv')
    material = build_thematic_material(coded_df, n=25)
    n_used = len(material.split('\n'))

    print(f"Sending {n_used} coded posts to {OLLAMA_MODEL} for thematic analysis...")

    try:
        raw = requests.post(
            OLLAMA_URL,
            json={'model': OLLAMA_MODEL, 'prompt': THEMATIC_PROMPT + material,
                  'stream': False, 'format': 'json'},
            timeout=300,
        )
        raw.raise_for_status()
        result = json.loads(raw.json()['response'])
        themes = result.get('themes', [])
    except Exception as e:
        print(f"Error: {e}")
        themes = []

    if themes:
        md = render_thematic_markdown(themes)
        thematic_path = OUTPUT_DIR / 'thematic_analysis.md'
        thematic_path.write_text(md, encoding='utf-8')
        print(f"Saved to: {thematic_path}\n")
        from IPython.display import Markdown
        display(Markdown(md))
    else:
        print("No themes returned — check Ollama logs or reduce input size.")
else:
    print('Skipping — Ollama server not running.')